In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import time

In [2]:
class RandomDataset(Dataset):
    def __init__(self, num_samples: int):
        self.img = torch.randn(num_samples, 1, 512, 512)
        self.scalar = torch.randn(num_samples, 1)
        self.label = torch.randn(num_samples, 1)

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        image = self.img[idx]
        scalar = self.scalar[idx]
        label = self.label[idx]
        return image, scalar, label

dataset = RandomDataset(1000)
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

# Test CPU

In [3]:
from CNetPlusScalar import CNetPlusScalar
model = CNetPlusScalar()
#model.load_state_dict(torch.load('pre_trained_w.pt'))

In [4]:
model.eval()
with torch.no_grad():
    inference_time = 0
    cpu_output = []
    for image, scalar, _ in dataloader:
        # Run inference for each image individually
        start_time = time.time()
        pred = model(image, scalar)
        inference_time += time.time() - start_time
        cpu_output.append(pred)

    # Convert list to numpy array
    cpu_output = np.array(cpu_output)

    # calculate average inference time and FPS
    num_images = len(cpu_output)
    avg_inference_time = inference_time / num_images
    fps = num_images / inference_time

    print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))
    cpu_inference_time = inference_time

The total inference time was 208.926369 seconds for 1000.
Average inference time per image: 0.208926 seconds
FPS: 4.79


# Test DPU

In [5]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("../vitisai_bitstream/dpu.bit")

In [6]:
overlay.load_model("zcu104_CNetPlusScalar.xmodel")

In [7]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn_img = tuple(inputTensors[0].dims)
shapeIn_scalar = tuple(inputTensors[1].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn_img[0])

output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn_img, dtype=np.float32, order="C"), np.empty(shapeIn_scalar, dtype=np.float32, order="C")]
data_img = input_data[0]
data_scalar = input_data[1]

In [8]:
inference_time = 0
vitisAI_output = []
for image, data_scalar, _ in dataloader:
    data = image.permute(0,2,3,1)
    start_time = time.time()
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)
    inference_time += time.time() - start_time
    vitisAI_output.append(output_data[0])

# Convert list to numpy array
vitisAI_output = np.array(vitisAI_output)

num_images = len(vitisAI_output)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print(f"Performance: {fps:.2f} FPS")
print(f"Speedup: {cpu_inference_time/inference_time:.2f}")

The total inference time was 6.115953 seconds for 1000.
Average inference time: 0.006115952730178833 s
Performance: 163.50682291338762 FPS
Speedup: 34.16


In [9]:
# Calculate MSE
mse = np.mean((cpu_output - vitisAI_output) ** 2)
print(f"Mean Squared Error between CPU and VitisAI outputs: {mse:.6f}")

Mean Squared Error between CPU and VitisAI outputs: 0.021986
